In [1]:
def compute_iou(heatmap, bbox, original_size=1024, target_size=320, threshold=0.2):
    """
    Compute IoU between thresholded Grad-CAM heatmap and radiologist bounding box.
    
    heatmap: numpy [320, 320] normalized 0-1
    bbox: (x, y, w, h) in original 1024x1024 coordinates
    threshold: top X% of activation becomes the predicted mask
    """
    x, y, w, h = bbox
    
    # Scale bbox from original size to target size
    scale = target_size / original_size
    x_s = int(x * scale)
    y_s = int(y * scale)
    w_s = max(1, int(w * scale))
    h_s = max(1, int(h * scale))
    
    # Create ground truth mask from bounding box
    gt_mask = np.zeros((target_size, target_size), dtype=np.uint8)
    gt_mask[y_s:y_s+h_s, x_s:x_s+w_s] = 1
    
    # Create predicted mask by thresholding heatmap
    pred_mask = (heatmap >= threshold).astype(np.uint8)
    
    # Compute IoU
    intersection = (gt_mask & pred_mask).sum()
    union = (gt_mask | pred_mask).sum()
    
    if union == 0:
        return 0.0
    
    return intersection / union

print("IoU function ready")

IoU function ready


In [2]:
from tqdm import tqdm

# Label name mapping between bbox file and our LABELS list
label_map = {
    'Atelectasis': 'Atelectasis',
    'Effusion': 'Effusion',
    'Cardiomegaly': 'Cardiomegaly',
    'Infiltrate': 'Infiltration',
    'Pneumonia': 'Pneumonia',
    'Pneumothorax': 'Pneumothorax',
    'Mass': 'Mass',
    'Nodule': 'Nodule'
}

results = {label: [] for label in label_map.keys()}

for idx, row in tqdm(bbox_df.iterrows(), total=len(bbox_df), desc="IoU validation"):
    finding = row['Finding Label']
    if finding not in label_map:
        continue
    
    our_label = label_map[finding]
    class_idx = LABELS.index(our_label)
    
    # Find image path
    image_path = None
    for folder in [f"images_{str(i).zfill(3)}" for i in range(1, 13)]:
        path = os.path.join(IMAGE_ROOT, folder, "images", row['Image Index'])
        if os.path.exists(path):
            image_path = path
            break
    
    if image_path is None:
        continue
    
    # Load and prepare image
    try:
        img = PILImage.open(image_path).convert('RGB')
        img_tensor = transform(img).unsqueeze(0)
        
        # Generate heatmap
        heatmap = gradcam.generate(img_tensor.clone(), class_idx)
        
        # Compute IoU
        bbox = (row['x'], row['y'], row['w'], row['h'])
        iou = compute_iou(heatmap, bbox)
        results[finding].append(iou)
    except Exception as e:
        continue

# Print results
print(f"\n{'Finding':<20} {'Count':<8} {'Mean IoU':<12} {'IoU>0.1':<12} {'IoU>0.25':<12}")
print("-" * 65)

all_ious = []
for finding, ious in results.items():
    if len(ious) == 0:
        continue
    ious_arr = np.array(ious)
    all_ious.extend(ious)
    mean_iou = ious_arr.mean()
    acc_01 = (ious_arr > 0.1).mean()
    acc_025 = (ious_arr > 0.25).mean()
    print(f"{finding:<20} {len(ious):<8} {mean_iou:<12.4f} {acc_01:<12.4f} {acc_025:<12.4f}")

print("-" * 65)
all_ious = np.array(all_ious)
print(f"{'Overall':<20} {len(all_ious):<8} {all_ious.mean():<12.4f} {(all_ious>0.1).mean():<12.4f} {(all_ious>0.25).mean():<12.4f}")

NameError: name 'bbox_df' is not defined